# Discharge anomaly
***

***Author:** Chus Casado Rodríguez*<br>
***Date:** 13-02-2025*<br>

**Introduction**<br>

This notebook computes the river discharge anomaly in the year 2024 based on the results of the GloFAS4 historical run. The reference period used as climatology spans from 1991 to 2020.

**Output**<br>
A NetCDF of river discharge anomaly in which pixels with catchment area smaller than 500 km2 are masked.

In [1]:
import numpy as np
import xarray as xr
import rioxarray as rxr
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from datetime import datetime

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: numpy._core.multiarray failed to import

## Configuration

In [ ]:
VAR = 'discharge'

# inputs
PATH_IN = Path('../data/GloFAS')
CLIMA_FILE = PATH_IN / VAR / f'{VAR}_avg_1991-2020.nc'
YEAR_FILE = PATH_IN / VAR / f'{VAR}_avg_2024.nc'
UPAREA_FILE = PATH_IN / 'static_maps' / 'upArea.nc'

# output
PATH_OUT = Path('../results') / VAR
PATH_PLOT = PATH_OUT / 'plots'
PATH_PLOT.mkdir(parents=True, exist_ok=True)

# minimum catchment area to be considered as river
MIN_AREA = 1e9 # m2  #5e8

map_variables = {
    'runoff': 'rowe',
    'discharge': 'dis24'
}

var_shortname = map_variables[VAR]

## Data

In [ ]:
# GloFAS upstream area map
uparea = xr.open_dataset(UPAREA_FILE)['Band1']
uparea.name = 'uparea'

# load climatology: average period 1991-2020
climatology = xr.open_dataset(CLIMA_FILE)[var_shortname]
climatology.close()

# load year of analysis: 2024
data = xr.open_dataset(YEAR_FILE)[var_shortname]
data.close()

## Anomaly

In [ ]:
# compute annual anomaly
anomaly = (data - climatology) # m3/s

# correct name and coordinates
anomaly.name = VAR
anomaly = anomaly.drop_vars('surface', errors='ignore')

# convert to rioxarray
anomaly = anomaly.rio.write_crs('EPSG:4326')

# find pixels with large enough catchment area
mask_area = uparea >= MIN_AREA

# make sure that latitude and longitude coordinates match with the anomaly dataset
mask_area = mask_area.sel(lat=slice(anomaly.lat.max(), anomaly.lat.min()))
mask_area['lat'] = anomaly.lat
mask_area['lon'] = anomaly.lon

# remove pixels with small catchment area
anomaly = anomaly.where(mask_area)

# add attributes
anomaly.attrs['long_name'] = 'river discharge anomaly'
anomaly.attrs['units'] = 'm3/s'
anomaly.attrs['climatology'] = '1991-2020'
anomaly.attrs['source'] = 'GloFASv4'
anomaly.attrs['crs'] = 'epsg:4326'
anomaly.attrs['author'] = 'Jesús Casado Rodríguez <jesus.casado-rodriguez@ec.europa.eu>'
anomaly.attrs['institution'] = 'Joint Research Centre - European Commission'
anomaly.attrs['history'] = 'Created {0}'.format(datetime.now().strftime("%B %d %Y %H:%M:%S"))

### Plot

#### Discharge anomaly (m3/s)

To produce the map of discharge anomaly I will convert the `xarray.DataArray` to `pandas.DataFrame` and plot it as points. The high resolution of GloFASv4 causes that the rivers are not noticeable when plotting it as a map.

In [ ]:
# round coordinates so they match
uparea['lon'] = uparea.lon.round(3)
uparea['lat'] = uparea.lat.round(3)
anomaly['lon'] = anomaly.lon.round(3)
anomaly['lat'] = anomaly.lat.round(3)

# merge discharge anomaly and upstream area
ds = xr.merge((uparea, anomaly))

# convert to DataFrame and remove missing values
df = ds.to_dataframe()[['uparea', 'discharge']]
df.dropna(how='any', inplace=True)
df.reset_index(inplace=True)
df.sort_values('uparea', ascending=True, inplace=True)

In [ ]:
# Define the colormap and normalization
boundaries = [-1e4, -1e3, -500, -250, -100, -50, -25, 25, 50, 100, 250, 500, 1e3, 1e4]
cmap = plt.cm.BrBG
ncolors = len(boundaries) - 1
colors = cmap(np.linspace(0, 1, ncolors))
colors[int(ncolors / 2), :] = [1., 1., 1., 1.]
cmap_discrete = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(boundaries, ncolors, extend='neither')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6), subplot_kw=dict(projection=ccrs.Robinson(central_longitude=-25)))#ccrs.PlateCarree()))

# Plot coastlines for continents
coastline = cfeature.NaturalEarthFeature(
    'physical', 'coastline', '50m', edgecolor='black', facecolor='none', linewidth=0.6
)
ax.add_feature(coastline)

sct = ax.scatter(
    df.lon,
    df.lat,
    c=df.discharge,
    s=.007,
    marker='s',
    cmap=cmap_discrete,
    norm=norm,
    transform=ccrs.PlateCarree()
);
plt.colorbar(
    sct,
    shrink=.5,
    aspect=30,
    orientation='horizontal', 
    pad=0.05,
    label=r'Anomalies from 1991-2020 ($m^3 \cdot s^{-1}$)',
    ticks=boundaries[1:-1]
)

# Add meridians and parallels
gridlines = ax.gridlines(draw_labels=False, linewidth=0.5, color='dimgray', alpha=0.5, linestyle='-')
ax.set_global();

plt.savefig(PATH_PLOT / 'map_anomaly_discharge.jpg', dpi=300, bbox_inches='tight')

#### Specific discharge (mm)

In [ ]:
# compute specific discharge (mm)
df['specific_discharge'] = df.discharge / df.uparea * 365 * 24 * 3600 * 1e3

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6), subplot_kw=dict(projection=ccrs.Robinson(central_longitude=-25)))

# Plot coastlines for continents
coastline = cfeature.NaturalEarthFeature(
    'physical', 'coastline', '50m', edgecolor='black', facecolor='none', linewidth=0.6
)
ax.add_feature(coastline)

sct = ax.scatter(
    df.lon,
    df.lat,
    c=df.specific_discharge,
    s=.007,
    marker='s',
    cmap=cmap_discrete,
    norm=norm,
    transform=ccrs.PlateCarree()
);
plt.colorbar(
    sct,
    shrink=.5,
    aspect=30,
    orientation='horizontal', 
    pad=0.05,
    label=r'Anomalies from 1991-2020 ($mm \cdot yr^{-1}$)',
    ticks=boundaries[1:-1]
)

# Add meridians and parallels
gridlines = ax.gridlines(draw_labels=False, linewidth=0.5, color='dimgray', alpha=0.5, linestyle='-')
ax.set_global();

plt.savefig(PATH_PLOT / 'map_anomaly_specific_discharge.jpg', dpi=300, bbox_inches='tight')

### Export


In [ ]:
# export
output_file = PATH_OUT / f'{VAR}_anomaly.nc'
anomaly.to_netcdf(output_file)
print(f'Map of river discharge anomaly in 2024 saved in:\t{output_file}')